# Testing Identity Propagation in Amazon Bedrock AgentCore

This tutorial demonstrates how JWT tokens propagate through AgentCore Runtime to Strands agents, enabling identity-aware authorization for multi-tenant applications.

## What You Will Learn

- Configure AgentCore Runtime with JWT authentication (Amazon Cognito)
- Extract user identity from JWT tokens using `BedrockAgentCoreContext`
- Understand the difference between Access Token and ID Token
- Implement multi-tenant data isolation in Strands agents
- Build role-based access control (RBAC) with user claims

## Architecture

```
┌─────────────┐    ID Token (JWT)    ┌──────────────────┐
│   Cognito   │ ─────────────────-─> │  AgentCore       │
│  User Pool  │   with custom attrs  │  Runtime         │
└─────────────┘                      │  - Validates JWT │
                                     │  - Forwards header│
                                     └──────────────────┘
                                              │
                                              │ Authorization header
                                              │ BedrockAgentCoreContext
                                              ▼
                                     ┌──────────────────┐
                                     │  Strands Agent   │
                                     │  - Decode JWT    │
                                     │  - Extract claims│
                                     │  - RBAC logic    │
                                     │  - Tenant filter │
                                     └──────────────────┘
```

## Key Concepts

### Token Types

| Token Type | Purpose | Contains | Use Case |
|------------|---------|----------|----------|
| **Access Token** | API authorization | `sub`, `scope`, `client_id` | Basic authentication |
| **ID Token** | User identity | `sub`, `aud`, **custom attributes** | **Multi-tenant apps** |

### Cognito Token Comparison

**Access Token:**
```json
{
  "sub": "user-uuid",
  "client_id": "3jea6...",
  "token_use": "access"
  // ❌ No aud field
  // ❌ No custom:tenant_id
  // ❌ No custom:user_role
}
```

**ID Token (we use this!):**
```json
{
  "sub": "user-uuid",
  "aud": "3jea6...",
  "cognito:username": "user-a",
  "custom:tenant_id": "tenant-a",  // ✓
  "custom:user_role": "admin",     // ✓
  "token_use": "id"
}
```

**Recommendation**: Use **ID Token** for multi-tenant authorization.

## Tutorial Details

| Component | Details |
|-----------|--------|
| Framework | Strands Agents |
| Model | Anthropic Claude 3.7 Sonnet |
| Auth Provider | Amazon Cognito |
| Complexity | Intermediate |
| Duration | 30-45 minutes |

## Prerequisites

- Python 3.10+
- AWS credentials configured
- Docker running (for local testing)
- Access to Amazon Bedrock models

## Step 1: Install Dependencies

In [26]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents pyyaml

In [27]:
import boto3
import json
import time
import base64
import yaml
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
cognito_client = boto3.client('cognito-idp', region_name=region)

print(f"Region: {region}")
print(f"Account ID: {account_id}")

Region: us-west-2
Account ID: 269550163595


## Step 2: Setup Amazon Cognito

Create a Cognito User Pool with custom attributes for multi-tenant testing.

In [28]:
user_pool_name = f"agentcore-identity-{int(time.time())}"

user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={'PasswordPolicy': {'MinimumLength': 8}},
    Schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True},
        {'Name': 'tenant_id', 'AttributeDataType': 'String', 'Mutable': True},
        {'Name': 'user_role', 'AttributeDataType': 'String', 'Mutable': True}
    ],
    AutoVerifiedAttributes=['email']
)

user_pool_id = user_pool_response['UserPool']['Id']

app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"{user_pool_name}-client",
    GenerateSecret=False,
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH']
)

client_id = app_client_response['UserPoolClient']['ClientId']
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

cognito_config = {
    'user_pool_id': user_pool_id,
    'client_id': client_id,
    'discovery_url': discovery_url
}

print(f"✓ User Pool: {user_pool_id}")
print(f"✓ Client ID: {client_id}")
print(f"✓ Discovery URL: {discovery_url}")

✓ User Pool: us-west-2_7uwf6Cotp
✓ Client ID: 4iodc3ipkcg2upn4ao6a6g9fqs
✓ Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_7uwf6Cotp/.well-known/openid-configuration


In [29]:
test_users = [
    {
        'username': 'user-a',
        'password': 'TempPass123!',
        'email': 'user-a@example.com',
        'tenant_id': 'tenant-a',
        'user_role': 'admin'
    },
    {
        'username': 'user-b',
        'password': 'TempPass123!',
        'email': 'user-b@example.com',
        'tenant_id': 'tenant-b',
        'user_role': 'user'
    }
]

created_users = []

for user in test_users:
    try:
        cognito_client.admin_create_user(
            UserPoolId=user_pool_id,
            Username=user['username'],
            UserAttributes=[
                {'Name': 'email', 'Value': user['email']},
                {'Name': 'email_verified', 'Value': 'true'},
                {'Name': 'custom:tenant_id', 'Value': user['tenant_id']},
                {'Name': 'custom:user_role', 'Value': user['user_role']}
            ],
            TemporaryPassword=user['password'],
            MessageAction='SUPPRESS'
        )
        cognito_client.admin_set_user_password(
            UserPoolId=user_pool_id,
            Username=user['username'],
            Password=user['password'],
            Permanent=True
        )
        created_users.append(user)
        print(f"  ✓ {user['username']} (Tenant: {user['tenant_id']}, Role: {user['user_role']})")
    except cognito_client.exceptions.UsernameExistsException:
        created_users.append(user)
        print(f"  ↻ {user['username']} already exists")

print(f"\n✓ Total users: {len(created_users)}")

  ✓ user-a (Tenant: tenant-a, Role: admin)
  ✓ user-b (Tenant: tenant-b, Role: user)

✓ Total users: 2


## Step 3: Create Identity-Aware Agent

This agent extracts user identity from JWT tokens and implements tenant isolation.

In [30]:
%%writefile identity_aware_agent.py
from strands import Agent
import json
import base64
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

def decode_jwt(token):
    """Decode JWT payload without verification"""
    try:
        parts = token.split('.')
        if len(parts) < 2:
            return None
        payload = parts[1]
        padding = 4 - (len(payload) % 4)
        if padding != 4:
            payload += '=' * padding
        return json.loads(base64.b64decode(payload))
    except Exception as e:
        print(f"JWT decode error: {e}")
        return None

def extract_user_identity(jwt_claims):
    """Extract user identity from JWT claims"""
    if not jwt_claims:
        return {
            'authenticated': False,
            'tenant_id': None,
            'user_role': None,
            'username': None,
            'sub': None
        }
    
    return {
        'authenticated': True,
        'tenant_id': jwt_claims.get('custom:tenant_id'),
        'user_role': jwt_claims.get('custom:user_role'),
        'username': jwt_claims.get('username') or jwt_claims.get('cognito:username'),
        'sub': jwt_claims.get('sub')
    }

# Initialize model and agent
model = BedrockModel(model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0")
agent = Agent(
    model=model,
    system_prompt="You're a helpful assistant with access to user context."
)

@app.entrypoint
def identity_aware_agent(payload):
    """Process requests with user identity awareness"""
    
    # Extract JWT from Authorization header
    headers = BedrockAgentCoreContext.get_request_headers() or {}
    auth_header = headers.get('authorization') or headers.get('Authorization')
    
    jwt_claims = None
    user_identity = None
    
    if auth_header:
        token = auth_header.replace('Bearer ', '').replace('bearer ', '')
        jwt_claims = decode_jwt(token)
        user_identity = extract_user_identity(jwt_claims)
    else:
        user_identity = extract_user_identity(None)
    
    # Log for CloudWatch
    print(f"User Identity: {json.dumps(user_identity, indent=2)}")
    
    # Process request
    user_input = payload.get("prompt", "")
    response = agent(user_input)
    agent_response = response.message['content'][0]['text']
    
    return {
        "response": agent_response,
        "user_identity": user_identity,
        "debug": {
            "request_id": BedrockAgentCoreContext.get_request_id(),
            "has_auth_header": bool(auth_header),
            "token_type": jwt_claims.get('token_use') if jwt_claims else None
        }
    }

if __name__ == "__main__":
    app.run()

Overwriting identity_aware_agent.py


## Step 4: Deploy Agent to AgentCore Runtime

Configure with JWT authentication and Authorization header forwarding.

In [31]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

# Configure agent
response = agentcore_runtime.configure(
    entrypoint="identity_aware_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="identity_test",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config['discovery_url'],
            "allowedAudience": [cognito_config['client_id']]  # For ID Token
        }
    }
)

print("✓ Basic configuration completed")

Entrypoint parsed: file=/Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity2/identity_aware_agent.py, bedrock_agentcore_name=identity_aware_agent
Memory configured with STM only
Configuring BedrockAgentCore agent: identity_test
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: identity_test_mem-gx6EYnC8r5
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity2/.dockerignore
Keeping 'identity_test' as default agent
Bedrock AgentCore configured: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity2/.bedrock_agentcore.yaml


✓ Basic configuration completed


### Add Authorization Header Forwarding

**Important**: The `Runtime.configure()` doesn't expose `request_header_configuration` parameter yet, so we modify the YAML directly.

In [32]:
# Read and modify the configuration YAML
with open('.bedrock_agentcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Add request header configuration
config['agents']['identity_test']['request_header_configuration'] = {
    'requestHeaderAllowlist': ['Authorization']  # Correct parameter name!
}

# Save modified configuration
with open('.bedrock_agentcore.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✓ Added Authorization header forwarding to configuration")

✓ Added Authorization header forwarding to configuration


In [33]:
# Verify configuration
print("Configuration Summary:")
print("="*60)
print(f"Agent Name: {config['agents']['identity_test']['name']}")
print(f"Discovery URL: {config['agents']['identity_test']['authorizer_configuration']['customJWTAuthorizer']['discoveryUrl']}")
print(f"Allowed Audience: {config['agents']['identity_test']['authorizer_configuration']['customJWTAuthorizer']['allowedAudience']}")
print(f"Request Headers: {config['agents']['identity_test']['request_header_configuration']['requestHeaderAllowlist']}")
print("="*60)

Configuration Summary:
Agent Name: identity_test
Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_7uwf6Cotp/.well-known/openid-configuration
Allowed Audience: ['4iodc3ipkcg2upn4ao6a6g9fqs']
Request Headers: ['Authorization']


### Launch Agent

In [34]:
print("Deploying agent (this may take several minutes)...\n")
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"\n✓ Agent deployed: {launch_result.agent_arn}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: identity_test
✅ MemoryManager initialized for region: us-west-2


Deploying agent (this may take several minutes)...



🔎 Retrieving memory resource with ID: identity_test_mem-gx6EYnC8r5...
  Found memory: identity_test_mem-gx6EYnC8r5
Found existing memory in cloud: identity_test_mem-gx6EYnC8r5
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'identity_test' to account 269550163595 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: identity_test
✅ ECR repository available: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-identity_test
Getting or creating execution role for agent: identity_test
Using AWS region: us-west-2, account ID: 269550163595
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d


✅ Reusing existing ECR repository: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-identity_test


✅ Reusing existing execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d
✅ Execution role available: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: identity_test
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5f0c170d0d
Reusing existing CodeBuild execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5f0c170d0d
Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: identity_test/source.zip
Updated CodeBuild project: bedrock-agentcore-identity_test-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 9.6s
🔄 DOWNLOAD_SOURCE started (total: 11s)
✅ DOWNLOAD_SOU


✓ Agent deployed: arn:aws:bedrock-agentcore:us-west-2:269550163595:runtime/identity_test-uUcXdvA7j8


In [35]:
print("Waiting for agent to be ready...")

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"  Status: {status}")

if status == 'READY':
    print(f"\n✓ Agent is ready: {status}")
else:
    raise Exception(f"Deployment failed: {status}")

✅ MemoryManager initialized for region: us-west-2


Waiting for agent to be ready...


🔎 Retrieving memory resource with ID: identity_test_mem-gx6EYnC8r5...
  Found memory: identity_test_mem-gx6EYnC8r5
Retrieved Bedrock AgentCore status for: identity_test



✓ Agent is ready: READY


## Step 5: Test Identity Propagation

### Test 1: Verify Authentication Required

In [36]:
print("TEST 1: Unauthenticated Request (should fail)")
print("="*60)

try:
    response = agentcore_runtime.invoke({"prompt": "test"})
    print("✗ UNEXPECTED: Request succeeded without auth")
except Exception as e:
    if "AccessDeniedException" in str(e) or "403" in str(e):
        print("✓ EXPECTED: Authentication enforced")
    else:
        print(f"✗ Unexpected error: {e}")

✅ MemoryManager initialized for region: us-west-2


TEST 1: Unauthenticated Request (should fail)


Memory is active, proceeding with invoke


✓ EXPECTED: Authentication enforced


### Test 2: Authenticate Users

In [37]:
def get_fresh_tokens(username, password):
    """Get fresh tokens from Cognito"""
    response = cognito_client.initiate_auth(
        ClientId=cognito_config['client_id'],
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

print("Authenticating users...")
user_tokens = {}

for user in created_users:
    tokens = get_fresh_tokens(user['username'], user['password'])
    user_tokens[user['username']] = tokens
    print(f"  ✓ {user['username']}")

print(f"\n✓ Authenticated {len(user_tokens)} users")

Authenticating users...
  ✓ user-a
  ✓ user-b

✓ Authenticated 2 users


### Test 3: Compare Access Token vs ID Token

In [38]:
def decode_and_display_token(token, token_name):
    parts = token.split('.')
    payload = parts[1]
    padding = 4 - (len(payload) % 4)
    if padding != 4:
        payload += '=' * padding
    claims = json.loads(base64.b64decode(payload))
    
    print(f"\n{token_name}:")
    print("-"*40)
    print(f"  token_use: {claims.get('token_use')}")
    print(f"  aud: {claims.get('aud', 'NOT FOUND')}")
    print(f"  client_id: {claims.get('client_id', 'NOT FOUND')}")
    print(f"  sub: {claims.get('sub')[:20]}...")
    print(f"  username: {claims.get('username') or claims.get('cognito:username', 'NOT FOUND')}")
    print(f"  custom:tenant_id: {claims.get('custom:tenant_id', 'NOT FOUND')}")
    print(f"  custom:user_role: {claims.get('custom:user_role', 'NOT FOUND')}")
    
    return claims

print("TOKEN COMPARISON: Access Token vs ID Token")
print("="*60)

user_a_tokens = user_tokens['user-a']
access_claims = decode_and_display_token(user_a_tokens['AccessToken'], 'Access Token')
id_claims = decode_and_display_token(user_a_tokens['IdToken'], 'ID Token')

print("\n" + "="*60)
print("CONCLUSION:")
print("  Access Token:")
print("    - Has client_id, no aud")
print("    - NO custom attributes")
print("\n  ID Token:")
print("    - Has aud (matches client_id)")
print("    - HAS custom attributes (tenant_id, user_role)")
print("    - ✓ USE THIS for multi-tenant apps")
print("="*60)

TOKEN COMPARISON: Access Token vs ID Token

Access Token:
----------------------------------------
  token_use: access
  aud: NOT FOUND
  client_id: 4iodc3ipkcg2upn4ao6a6g9fqs
  sub: 08a1c360-d051-7083-9...
  username: user-a
  custom:tenant_id: NOT FOUND
  custom:user_role: NOT FOUND

ID Token:
----------------------------------------
  token_use: id
  aud: 4iodc3ipkcg2upn4ao6a6g9fqs
  client_id: NOT FOUND
  sub: 08a1c360-d051-7083-9...
  username: user-a
  custom:tenant_id: tenant-a
  custom:user_role: admin

CONCLUSION:
  Access Token:
    - Has client_id, no aud
    - NO custom attributes

  ID Token:
    - Has aud (matches client_id)
    - HAS custom attributes (tenant_id, user_role)
    - ✓ USE THIS for multi-tenant apps


### Test 4: Identity Propagation with User A

In [39]:
print("TEST 4: User A (Tenant A Admin) - Identity Propagation")
print("="*60)

# Get fresh ID Token
fresh_tokens = get_fresh_tokens('user-a', 'TempPass123!')
id_token = fresh_tokens['IdToken']

try:
    response = agentcore_runtime.invoke(
        {"prompt": "What is 2+2?"},
        bearer_token=id_token
    )
    
    # Parse response
    resp_data = response.get('response')
    if isinstance(resp_data, str):
        try:
            parsed = json.loads(resp_data)
            agent_resp = parsed['response']
            identity = parsed['user_identity']
            debug = parsed.get('debug', {})
        except:
            agent_resp = resp_data
            identity = response.get('user_identity', {})
            debug = response.get('debug', {})
    else:
        agent_resp = resp_data
        identity = response.get('user_identity', {})
        debug = response.get('debug', {})
    
    print(f"\nAgent Response: {agent_resp}")
    
    print(f"\nUser Identity:")
    print(f"  Authenticated: {identity.get('authenticated')}")
    print(f"  Username: {identity.get('username')}")
    print(f"  Tenant ID: {identity.get('tenant_id')}")
    print(f"  User Role: {identity.get('user_role')}")
    print(f"  Sub: {identity.get('sub', 'N/A')[:20]}...")
    
    print(f"\nDebug Info:")
    print(f"  Has Auth Header: {debug.get('has_auth_header')}")
    print(f"  Token Type: {debug.get('token_type')}")
    print(f"  Request ID: {debug.get('request_id')}")
    
    if (identity.get('authenticated') and 
        identity.get('tenant_id') == 'tenant-a' and 
        identity.get('user_role') == 'admin'):
        print("\n✓ SUCCESS: Identity propagation working!")
        print(f"  JWT → AgentCore → BedrockAgentCoreContext → Agent")
        print(f"  User '{identity.get('username')}' authenticated as tenant '{identity.get('tenant_id')}' admin")
    else:
        print("\n✗ FAILED: Identity values incorrect")
        
except Exception as e:
    print(f"\n✗ ERROR: {e}")
    print("\nTroubleshooting:")
    print("  1. Verify agent is READY")
    print("  2. Check requestHeaderAllowlist in YAML")
    print("  3. Confirm ID Token is being used (not Access Token)")
    print("  4. Review CloudWatch Logs for agent errors")

TEST 4: User A (Tenant A Admin) - Identity Propagation

Agent Response: The answer to 2+2 is 4.

User Identity:
  Authenticated: True
  Username: user-a
  Tenant ID: tenant-a
  User Role: admin
  Sub: 08a1c360-d051-7083-9...

Debug Info:
  Has Auth Header: True
  Token Type: id
  Request ID: 8294a301-1ef2-4bc1-9597-6c017373b761

✓ SUCCESS: Identity propagation working!
  JWT → AgentCore → BedrockAgentCoreContext → Agent
  User 'user-a' authenticated as tenant 'tenant-a' admin


### Test 5: Multi-Tenant Isolation

In [40]:
print("TEST 5: Multi-Tenant Isolation")
print("="*60)

results = []

for user in created_users:
    try:
        fresh_tokens = get_fresh_tokens(user['username'], user['password'])
        id_token = fresh_tokens['IdToken']
        
        response = agentcore_runtime.invoke(
            {"prompt": "Hello"},
            bearer_token=id_token
        )
        
        resp_data = response.get('response')
        if isinstance(resp_data, str):
            try:
                parsed = json.loads(resp_data)
                identity = parsed['user_identity']
            except:
                identity = response.get('user_identity', {})
        else:
            identity = response.get('user_identity', {})
        
        results.append({
            'username': user['username'],
            'expected_tenant': user['tenant_id'],
            'actual_tenant': identity.get('tenant_id'),
            'expected_role': user['user_role'],
            'actual_role': identity.get('user_role'),
            'authenticated': identity.get('authenticated')
        })
    except Exception as e:
        print(f"  ✗ {user['username']}: {e}")
        results.append({'username': user['username'], 'error': str(e)})

# Display results
print(f"\n{'User':<10} {'Expected':<12} {'Actual':<12} {'Expected Role':<12} {'Actual Role':<12} {'Status'}")
print("-"*80)

all_success = True
for r in results:
    if 'error' in r:
        print(f"{r['username']:<10} {'ERROR'}")
        all_success = False
    else:
        tenant_match = "✓" if r['expected_tenant'] == r['actual_tenant'] else "✗"
        role_match = "✓" if r['expected_role'] == r['actual_role'] else "✗"
        status = "✓" if (r['expected_tenant'] == r['actual_tenant'] and 
                        r['expected_role'] == r['actual_role'] and 
                        r['authenticated']) else "✗"
        
        print(f"{r['username']:<10} {r['expected_tenant']:<12} {r['actual_tenant']:<12} "
              f"{r['expected_role']:<12} {r['actual_role']:<12} {status}")
        
        if status == "✗":
            all_success = False

print("\n" + "="*80)
if all_success:
    print("✓ SUCCESS: Multi-tenant isolation verified!")
    print("  Each user correctly identified with their tenant and role")
else:
    print("✗ FAILED: Issues detected in tenant isolation")

TEST 5: Multi-Tenant Isolation

User       Expected     Actual       Expected Role Actual Role  Status
--------------------------------------------------------------------------------
user-a     tenant-a     tenant-a     admin        admin        ✓
user-b     tenant-b     tenant-b     user         user         ✓

✓ SUCCESS: Multi-tenant isolation verified!
  Each user correctly identified with their tenant and role


## Step 6: Advanced Example - Role-Based Access Control

This example shows how to implement RBAC in Strands agents based on JWT claims.

In [41]:
%%writefile rbac_agent_example.py
from strands import Agent, tool
import json
import base64
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Mock data by tenant
TENANT_DATA = {
    'tenant-a': {'orders': ['order-a1', 'order-a2'], 'revenue': 50000},
    'tenant-b': {'orders': ['order-b1'], 'revenue': 30000}
}

def decode_jwt(token):
    try:
        parts = token.split('.')
        payload = parts[1]
        padding = 4 - (len(payload) % 4)
        if padding != 4:
            payload += '=' * padding
        return json.loads(base64.b64decode(payload))
    except:
        return None

def create_tenant_tools(tenant_id, user_role):
    """Create tools scoped to tenant and filtered by role"""
    
    @tool
    def get_orders() -> str:
        f"""Get orders for tenant {tenant_id}"""
        orders = TENANT_DATA.get(tenant_id, {}).get('orders', [])
        return f"Orders for {tenant_id}: {', '.join(orders)}"
    
    @tool
    def get_revenue() -> str:
        f"""Get revenue for tenant {tenant_id} (admin only)"""
        if user_role != 'admin':
            return "Permission denied: Admin role required"
        revenue = TENANT_DATA.get(tenant_id, {}).get('revenue', 0)
        return f"Revenue for {tenant_id}: ${revenue}"
    
    # Return tools based on role
    if user_role == 'admin':
        return [get_orders, get_revenue]
    else:
        return [get_orders]  # Read-only for non-admin

@app.entrypoint
def rbac_agent(payload):
    # Extract JWT
    headers = BedrockAgentCoreContext.get_request_headers() or {}
    auth = headers.get('Authorization')
    
    if not auth:
        return {"error": "Unauthorized"}
    
    claims = decode_jwt(auth.replace('Bearer ', ''))
    if not claims:
        return {"error": "Invalid token"}
    
    tenant_id = claims.get('custom:tenant_id')
    user_role = claims.get('custom:user_role')
    username = claims.get('cognito:username')
    
    # Create tenant-scoped and role-filtered tools
    tools = create_tenant_tools(tenant_id, user_role)
    
    # Create agent with appropriate tools
    model = BedrockModel(model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0")
    agent = Agent(
        model=model,
        tools=tools,
        system_prompt=f"You are assisting {username} from {tenant_id}. "
                     f"User role: {user_role}. Only access data for this tenant."
    )
    
    response = agent(payload.get("prompt", ""))
    
    return {
        "response": response.message['content'][0]['text'],
        "user_context": {
            "username": username,
            "tenant_id": tenant_id,
            "role": user_role,
            "available_tools": [t.__name__ for t in tools]
        }
    }

if __name__ == "__main__":
    app.run()

Overwriting rbac_agent_example.py


## Summary

In [42]:
print("\n" + "="*80)
print("IDENTITY PROPAGATION TUTORIAL SUMMARY")
print("="*80)

print("\n📊 Test Results:")
print(f"  ✓ Users authenticated: {len(user_tokens)}")
print(f"  ✓ Tenants verified: {len(set(u['tenant_id'] for u in created_users))}")
print(f"  ✓ Agent status: {status}")

print("\n🔑 Key Findings:")
print("  1. JWT tokens propagate: Client → AgentCore → Strands Agent")
print("  2. ID Token contains custom attributes (tenant_id, user_role)")
print("  3. Access Token does NOT contain custom attributes")
print("  4. AgentCore validates JWT (discoveryUrl, allowedAudience)")
print("  5. Authorization header forwarding requires requestHeaderAllowlist")
print("  6. BedrockAgentCoreContext.get_request_headers() provides access")
print("  7. Strands agents can implement tenant isolation and RBAC")

print("\n🏗️ Architecture:")
print("  Client → Bearer Token (ID Token)")
print("         → AgentCore validates JWT")
print("         → Authorization header forwarded")
print("         → BedrockAgentCoreContext")
print("         → Strands Agent decodes JWT")
print("         → Extract claims (tenant_id, user_role)")
print("         → Implement authorization logic")

print("\n💡 Best Practices:")
print("  ✓ Use ID Token for multi-tenant applications")
print("  ✓ Configure allowedAudience to match ID Token's aud claim")
print("  ✓ Add requestHeaderAllowlist: ['Authorization']")
print("  ✓ Extract identity in agent using BedrockAgentCoreContext")
print("  ✓ Implement tenant-based data isolation in Strands")
print("  ✓ Filter tools based on user roles")
print("  ✓ Log user context for audit trails")

print("\n⚠️  Important Notes:")
print("  - ID Token typically used for client authentication (non-standard for API)")
print("  - AgentCore allows ID Token as Bearer for efficiency")
print("  - Eliminates need for token introspection calls")
print("  - Custom attributes only in ID Token (Cognito limitation)")
print("  - requestHeaderAllowlist: singular 'Allowlist', not 'Headers'")

print("\n📚 Configuration Reference:")
print("""  
  authorizer_configuration:
    customJWTAuthorizer:
      discoveryUrl: <cognito-discovery-url>
      allowedAudience: [<client-id>]  # For ID Token
  
  request_header_configuration:
    requestHeaderAllowlist: ['Authorization']  # Required!
""")

print("="*80)
print("✓ Tutorial Complete!")
print("="*80)


IDENTITY PROPAGATION TUTORIAL SUMMARY

📊 Test Results:
  ✓ Users authenticated: 2
  ✓ Tenants verified: 2
  ✓ Agent status: ✓

🔑 Key Findings:
  1. JWT tokens propagate: Client → AgentCore → Strands Agent
  2. ID Token contains custom attributes (tenant_id, user_role)
  3. Access Token does NOT contain custom attributes
  4. AgentCore validates JWT (discoveryUrl, allowedAudience)
  5. Authorization header forwarding requires requestHeaderAllowlist
  6. BedrockAgentCoreContext.get_request_headers() provides access
  7. Strands agents can implement tenant isolation and RBAC

🏗️ Architecture:
  Client → Bearer Token (ID Token)
         → AgentCore validates JWT
         → Authorization header forwarded
         → BedrockAgentCoreContext
         → Strands Agent decodes JWT
         → Extract claims (tenant_id, user_role)
         → Implement authorization logic

💡 Best Practices:
  ✓ Use ID Token for multi-tenant applications
  ✓ Configure allowedAudience to match ID Token's aud claim
  

## Cleanup (Optional)

Uncomment and run to delete resources.

In [43]:
# # Delete AgentCore Runtime
# agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
# agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
# print("✓ Deleted AgentCore Runtime")

# # Delete ECR repository
# ecr_client = boto3.client('ecr', region_name=region)
# repo_name = launch_result.ecr_uri.split('/')[1]
# ecr_client.delete_repository(repositoryName=repo_name, force=True)
# print("✓ Deleted ECR repository")

# # Delete Cognito User Pool
# cognito_client.delete_user_pool(UserPoolId=cognito_config['user_pool_id'])
# print("✓ Deleted Cognito User Pool")

# Congratulations!

You've successfully implemented identity-aware authorization in Amazon Bedrock AgentCore!

## What You Learned

1. **JWT Token Flow**: How ID Tokens propagate through AgentCore to Strands agents
2. **Token Differences**: Access Token vs ID Token and when to use each
3. **Identity Extraction**: Using `BedrockAgentCoreContext` to access JWT claims
4. **Multi-Tenant Isolation**: Implementing tenant-based data filtering
5. **Role-Based Access Control**: Dynamic tool filtering based on user roles

## Next Steps

For production deployments:
1. Implement comprehensive audit logging with tenant context
2. Add database-level row security policies
3. Set up monitoring for unauthorized access attempts
4. Consider token refresh strategies for long-running sessions
5. Implement rate limiting per tenant

## References

- [AgentCore Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore.html)
- [ID Token vs Access Token](https://auth0.com/blog/id-token-access-token-what-is-the-difference/)
- [Strands Agents](https://github.com/awslabs/strands)
- [Cognito Custom Attributes](https://docs.aws.amazon.com/cognito/latest/developerguide/user-pool-settings-attributes.html)